<a href="https://colab.research.google.com/github/RahulReddyKota/Intern_Challenge/blob/main/Rahul_SignalDesk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
"""
SignalDesk weekly check
=======================

One question: can we trust this week's export, and did the prompt change help?

The script quarantines rows it cannot trust BEFORE reporting any headline
number, then shows how much the answer depends on that quarantine. If the
verdict flips when the quarantined rows go back in, the honest answer is
"we don't know yet" -- and the script says so rather than picking a side.

Usage (terminal):
    python signaldesk_check.py --csv product_usage_events.csv

Usage (notebook / Colab cell):
    from signaldesk_check import main
    main(["--csv", "product_usage_events.csv"])
"""

import argparse
import os
import sys

import numpy as np
import pandas as pd

# --- Config ------------------------------------------------------------------
# All thresholds live here so a teammate can argue with the numbers without
# reading the code. None of them are principled; they are starting points.

OUTLIER_HIGH   = 2.0   # sessions > 2.0x its own series median -> suspicious
OUTLIER_LOW    = 0.5   # sessions < 0.5x its own series median -> suspicious
MIN_SERIES_LEN = 4     # fewer days than this and outlier logic means nothing
THIN_SESSIONS  = 30    # rates below this denominator get a caveat marker
FLIP_TOL       = 0.005 # sign changes smaller than this are not a real flip

W = 78
IND = " " * 14


def rule(c="-"):
    return c * W


def wrap(text, width):
    words, line, out = text.split(), "", []
    for w in words:
        if len(line) + len(w) + 1 > width:
            out.append(line); line = w
        else:
            line = f"{line} {w}".strip()
    if line:
        out.append(line)
    return out


def pct(x, spec=".0%"):
    """NaN-safe percent formatting."""
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else format(x, spec)


# --- Load --------------------------------------------------------------------

def load(path):
    if not os.path.exists(path):
        sys.exit(f"ERROR: CSV not found at '{path}'. Pass the correct path with --csv.")
    raw = pd.read_csv(path)

    required = {"date", "team", "workflow", "source", "sessions", "completed",
                "accepted_output", "flagged_for_review", "avg_minutes_saved",
                "median_confidence", "user_rating", "notes"}
    missing = required - set(raw.columns)
    if missing:
        sys.exit(f"ERROR: CSV is missing expected column(s): {', '.join(sorted(missing))}")

    df = raw.copy()
    fixes = []

    for col in ("team", "workflow", "source"):
        before = df[col].dropna().nunique()
        # 'string' dtype keeps nulls as nulls; plain astype(str) would turn
        # NaN into the literal text "nan" and silently invent a category.
        df[col] = df[col].astype("string").str.strip()
        if col == "team":
            df[col] = df[col].str.title()
        after = df[col].dropna().nunique()
        if after < before:
            fixes.append(f"{col}: {before} distinct values collapsed to {after} "
                         f"after trimming and case-folding")

    df["date"] = pd.to_datetime(df["date"])

    # pandas already reads 'n/a', 'N/A', '' etc. as null. Coercing again is
    # cheap insurance against a marker it does not know ('none', '-', 'TBD').
    for col in ("avg_minutes_saved", "median_confidence", "user_rating"):
        coerced = pd.to_numeric(df[col], errors="coerce")
        extra = int(coerced.isna().sum() - df[col].isna().sum())
        if extra:
            fixes.append(f"{col}: {extra} unparseable value(s) coerced to null")
        df[col] = coerced

    df["notes"] = df["notes"].fillna("").astype(str)
    return df, fixes, len(raw)


# --- Hazard scan -------------------------------------------------------------

def scan(df):
    df = df.copy()
    df["disposition"] = "keep"
    findings = []

    def add(level, name, sessions, detail):
        findings.append((level, name, sessions, detail))

    # EXCLUDE -- exact duplicates. Plumbing, not judgement.
    key = [c for c in df.columns if c not in ("notes", "disposition")]
    dupes = df.duplicated(subset=key, keep="first")
    for _, r in df[dupes].iterrows():
        add("EXCLUDE", "duplicate row", int(r.sessions),
            f"{r.date.date()} {r.team}/{r.workflow}/{r.source} is identical to an "
            f"earlier row in every field except notes.")
    df.loc[dupes, "disposition"] = "exclude"

    # QUARANTINE -- volume anomalies, found structurally. Deliberately NOT keyed
    # off the notes column: a check that depends on someone typing "demo account"
    # is a check that silently stops working.
    live = df[df.disposition == "keep"]
    for (team, wf, src), g in live.groupby(["team", "workflow", "source"]):
        if len(g) < MIN_SERIES_LEN:
            continue
        med = g.sessions.median()
        if med <= 0:
            continue
        ratio = g.sessions / med
        for idx, r in g[(ratio > OUTLIER_HIGH) | (ratio < OUTLIER_LOW)].iterrows():
            kind = "spike" if r.sessions > med else "collapse"
            df.loc[idx, "disposition"] = "quarantine"
            note = f" Notes say: '{r.notes}'." if r.notes else ""
            add("QUARANTINE", f"volume {kind}", int(r.sessions),
                f"{r.date.date()} {team}/{wf}/{src}: {int(r.sessions)} sessions against a "
                f"series median of {med:g} ({r.sessions / med:.1f}x).{note}")

    # WARN -- completeness, measured AFTER quarantine, because quarantining a row
    # can itself leave a day missing a series.
    trusted = df[df.disposition == "keep"]
    all_series = set(map(tuple, df[["team", "workflow", "source"]].drop_duplicates().values))
    for day, g in trusted.groupby("date"):
        present = set(map(tuple, g[["team", "workflow", "source"]].values))
        missing = sorted(all_series - present)
        if missing:
            add("WARN", "incomplete day", 0,
                f"{day.date()}: {len(present)}/{len(all_series)} series usable. Missing "
                + ", ".join("/".join(m) for m in missing)
                + ". Day-level totals for this date are not comparable to other days.")

    for col in ("median_confidence", "user_rating", "avg_minutes_saved"):
        n = int(trusted[col].isna().sum())
        if n:
            add("WARN", "missing values", 0,
                f"{col}: {n} null(s) among kept rows, so this metric has a different "
                f"denominator than the counts next to it.")

    bad = trusted[(trusted.completed > trusted.sessions)
                  | (trusted.accepted_output > trusted.completed)]
    if len(bad):
        add("WARN", "impossible counts", 0,
            f"{len(bad)} row(s) where accepted > completed or completed > sessions. "
            f"Stop and check the export job.")

    order = {"EXCLUDE": 0, "QUARANTINE": 1, "WARN": 2}
    findings.sort(key=lambda f: (order[f[0]], f[3]))
    return df, findings


# --- Metrics -----------------------------------------------------------------

def agg(d):
    """Session-weighted. Never a mean of per-row rates: that would let a
    6-session day count the same as a 140-session one. Returns None when
    there is nothing to aggregate, so callers must check."""
    if len(d) == 0:
        return None
    s = d.sessions.sum()
    if s == 0:
        return None
    c, a, f = d.completed.sum(), d.accepted_output.sum(), d.flagged_for_review.sum()
    m = d.dropna(subset=["avg_minutes_saved"])
    m = m[m.sessions > 0]
    return {
        "sessions": int(s),
        "completion": c / s,
        "acc_sess": a / s,
        "acc_compl": a / c if c else np.nan,
        "flag_compl": f / c if c else np.nan,
        "minutes": np.average(m.avg_minutes_saved, weights=m.sessions) if len(m) else np.nan,
    }


def change_date(df, override):
    if override:
        try:
            return pd.to_datetime(override)
        except (ValueError, TypeError):
            sys.exit(f"ERROR: --change-date '{override}' is not a date. Use YYYY-MM-DD.")
    hits = df[df.notes.str.contains("new prompt", case=False, na=False)]
    return hits.date.min() if len(hits) else None


# --- Report ------------------------------------------------------------------

def main(args_list=None):
    ap = argparse.ArgumentParser(description="Weekly trust check for the SignalDesk usage export.")
    ap.add_argument("--csv", required=True)
    ap.add_argument("--out", default="analyzed_results.csv",
                    help="Path to output the annotated CSV file.")
    ap.add_argument("--change-date", default=None,
                    help="YYYY-MM-DD. Defaults to the first date whose notes mention a new prompt.")
    args = ap.parse_args(args_list)

    df, fixes, n_raw = load(args.csv)
    df, findings = scan(df)

    trusted  = df[df.disposition == "keep"]
    deduped  = df[df.disposition != "exclude"]
    n_ex     = int((df.disposition == "exclude").sum())
    n_q      = int((df.disposition == "quarantine").sum())

    print(f"\n{rule('=')}")
    print("SIGNALDESK WEEKLY CHECK")
    print(f"{args.csv}")
    print(f"{df.date.min().date()} to {df.date.max().date()}  |  {n_raw} rows in")
    print(rule("="))

    # 1 -----------------------------------------------------------------
    print("\n1. HAZARD SCAN")
    print(rule())
    for f in fixes:
        print(f"  {'FIXED':<11}{f}")
    for level, name, sess, detail in findings:
        suffix = f"  [{sess} sessions]" if sess else ""
        print(f"  {level:<11}{name}{suffix}")
        for line in wrap(detail, W - len(IND)):
            print(f"{IND}{line}")
    if not fixes and not findings:
        print("  Nothing flagged. Either the export is clean or the checks are blind.")
    print(f"\n  -> {len(trusted)}/{n_raw} rows trusted. {n_ex} excluded, {n_q} quarantined.")
    print("     Excluded rows are gone. Quarantined rows are held out of section 2,")
    print("     stress-tested in section 3, and still used as evidence in section 4.")

    # 2 -----------------------------------------------------------------
    print("\n2. THE WEEK, TRUSTED ROWS ONLY")
    print(rule())
    print(f"  {'workflow':<22}{'sess':>6}{'compl':>8}{'acc/sess':>10}{'acc/compl':>11}{'flag/compl':>12}")
    for wf, g in trusted.groupby("workflow"):
        m = agg(g)
        if m is None:
            continue
        print(f"  {wf:<22}{m['sessions']:>6}{pct(m['completion']):>8}{pct(m['acc_sess']):>10}"
              f"{pct(m['acc_compl']):>11}{pct(m['flag_compl']):>12}")
    print("\n  By source, because manual and automated intake are different products:")
    rows = sorted(trusted.groupby(["workflow", "source"]), key=lambda kv: -kv[1].sessions.sum())
    for (wf, src), g in rows:
        m = agg(g)
        if m is None:
            continue
        flag = "   <- thin denominator, treat rate as noise" if m["sessions"] < THIN_SESSIONS else ""
        print(f"    {wf + ' / ' + src:<34}{m['sessions']:>5} sess   acc/sess {pct(m['acc_sess']):>4}{flag}")

    # 3 -----------------------------------------------------------------
    cd = change_date(df, args.change_date)
    print(f"\n3. DID THE {cd.date() if cd is not None else '???'} PROMPT CHANGE HELP?")
    print(rule())
    if cd is None:
        print("  No prompt-change date in notes and none supplied. Skipped.")
    elif cd <= df.date.min() or cd > df.date.max():
        print(f"  Change date {cd.date()} leaves no data on one side of the split.")
        print("  Nothing to compare. Check --change-date or the notes column.")
    else:
        print(f"  {'rows counted':<32}{'pre':>7}{'post':>8}{'delta':>9}{'flag delta':>13}")
        deltas = []
        for label, data in [("as exported", df),
                            ("exact duplicates removed", deduped),
                            ("+ quarantined rows removed", trusted)]:
            a, b = agg(data[data.date < cd]), agg(data[data.date >= cd])
            if a is None or b is None:
                print(f"  {label:<32}{'-- empty on one side of the split --':>37}")
                continue
            d = b["acc_sess"] - a["acc_sess"]
            deltas.append(d)
            print(f"  {label:<32}{pct(a['acc_sess'], '.1%'):>7}{pct(b['acc_sess'], '.1%'):>8}"
                  f"{pct(d, '+.1%'):>9}{pct(b['flag_compl'] - a['flag_compl'], '+.1%'):>13}")

        moved = df[df.disposition != "keep"]
        post_sessions = df[df.date >= cd].sessions.sum()
        print()
        if not deltas:
            print("  VERDICT: no comparable rows on both sides of the change date.")
        elif min(deltas) < -FLIP_TOL and max(deltas) > FLIP_TOL:
            share = moved.sessions.sum() / post_sessions if post_sessions else np.nan
            print("  VERDICT: UNRESOLVED. The sign of the effect is decided entirely by")
            print(f"  whether {len(moved)} of {n_raw} rows are counted -- {int(moved.sessions.sum())} sessions, "
                  f"{pct(share)} of the")
            print("  post-change period. Do not report a win or a regression off this")
            print("  export. Resolve the quarantine first, then re-run.")
        elif max(deltas) < 0:
            print("  VERDICT: acceptance is DOWN post-change under every cleaning rule.")
        else:
            print("  VERDICT: acceptance is UP post-change under every cleaning rule.")

        n_pre = df[df.date < cd].date.nunique()
        n_post = df[df.date >= cd].date.nunique()
        print(f"\n  Even the bottom row is not a causal estimate: {n_pre} pre days vs {n_post} post,")
        print("  no control group, and nothing here rules out other changes that")
        print("  shipped in the same window.")

    # 4 -----------------------------------------------------------------
    print("\n4. IS median_confidence WORTH ANYTHING?")
    print(rule())
    print("  Model self-report. Never reported as quality -- only checked against")
    print("  outcomes. Spearman rather than Pearson: with a week of daily points a")
    print("  single spike can flip a linear correlation's sign while the ranks barely")
    print("  move. Quarantined rows are kept here on purpose -- a bad day is evidence.\n")
    print(f"    {'series':<38}{'sess':>6}{'rank corr':>11}")
    series = sorted(deduped.groupby(["workflow", "source"]), key=lambda kv: -kv[1].sessions.sum())
    diverging, corrs = [], []
    for (wf, src), g in series:
        g = g.dropna(subset=["median_confidence"])
        g = g[g.sessions > 0]
        if len(g) < MIN_SERIES_LEN:
            print(f"    {wf + ' / ' + src:<38}{int(g.sessions.sum()):>6}{'too short':>11}")
            continue
        acc = (g.accepted_output / g.sessions).reset_index(drop=True)
        r = g.median_confidence.reset_index(drop=True).corr(acc, method="spearman")
        if np.isnan(r):
            print(f"    {wf + ' / ' + src:<38}{int(g.sessions.sum()):>6}{'no variance':>11}")
            continue
        corrs.append((src, r))
        note = "  <- inverted" if r < -0.5 else ""
        if r < -0.5:
            diverging.append(f"{wf}/{src}")
        print(f"    {wf + ' / ' + src:<38}{int(g.sessions.sum()):>6}{r:>+11.2f}{note}")
    print()
    if corrs:
        manual = [r for (s, r) in corrs if s == "manual"]
        auto = [r for (s, r) in corrs if s != "manual"]
        if manual and auto and min(manual) > 0 > max(auto):
            print(f"  Every manual series is positive ({', '.join(f'{r:+.2f}' for r in sorted(manual, reverse=True))}) and every")
            print(f"  automated series is negative ({', '.join(f'{r:+.2f}' for r in sorted(auto, reverse=True))}). Clean split, but it is")
            print(f"  {len(corrs)} short daily series -- a pattern to go check, not a finding.")
        elif diverging:
            print(f"  Inverted: {', '.join(diverging)}.")
        valid = deduped[deduped.sessions > 0].copy()
        valid["acc_rate"] = valid.accepted_output / valid.sessions
        worst = valid.loc[valid.acc_rate.idxmin()]
        conf_note = ("the week's highest confidence"
                     if worst.median_confidence == valid.median_confidence.max()
                     else f"a confidence of")
        print(f"\n  Sharpest single case: {worst.date.date()} {worst.workflow}/{worst.source} posts")
        print(f"  {conf_note} ({worst.median_confidence:.2f}) on its worst acceptance "
              f"({worst.acc_rate:.0%}).")
        if worst.median_confidence == valid.median_confidence.max():
            print("  Whatever confidence is measuring, it is not that.")
    else:
        print("  Not enough history to say. Still not a quality metric.")

    # 5 -----------------------------------------------------------------
    print("\n5. WHAT THIS DOES NOT TELL YOU")
    print(rule())
    for a, b in [
        ("accepted_output means 'not reworked', not 'correct'.",
         "A confidently wrong output nobody checked counts as a success here."),
        ("flagged_for_review conflates bad output with strict review.",
         "A rise can mean quality fell or that policy tightened. Section 3's flag delta is directional at best."),
        ("avg_minutes_saved is estimated by the tool, not measured.",
         "Session-weighting a guess does not make it a measurement."),
        ("sessions are runs, not people.",
         "Nothing here says how many humans this reaches, or whether it is ten users or two hundred."),
        ("One week, no control group, and no baseline from before these workflows launched.",
         "The workflows serve different teams on different intake paths, so their acceptance rates are not competing scores. Do not rank them."),
    ]:
        print(f"  - {a}")
        for line in wrap(b, W - 6):
            print(f"    {line}")
    print()

    # 6 -----------------------------------------------------------------
    print("\n6. DATA EXPORT")
    print(rule())
    out_dir = os.path.dirname(os.path.abspath(args.out))
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(args.out, index=False)
    print(f"  Data exported successfully to {args.out}")
    return df


if __name__ == "__main__":
    # In Colab/Jupyter, sys.argv is populated with kernel launcher arguments
    # Checking for typical Jupyter entry point arguments helps avoid argparse errors
    if any("ipykernel" in arg or "colab_kernel_launcher.py" in arg for arg in sys.argv):
        main(["--csv", "product_usage_events.csv"])       # notebook/colab use
    elif len(sys.argv) > 1:
        main()                                            # normal CLI use
    else:
        main(["--csv", "product_usage_events.csv"])       # bare run



SIGNALDESK WEEKLY CHECK
product_usage_events.csv
2026-08-01 to 2026-08-07  |  41 rows in

1. HAZARD SCAN
------------------------------------------------------------------------------
  FIXED      team: 4 distinct values collapsed to 3 after trimming and case-folding
  EXCLUDE    duplicate row  [140 sessions]
              2026-08-05 Sales/Lead summary/email is identical to an earlier
              row in every field except notes.
  QUARANTINE volume spike  [140 sessions]
              2026-08-05 Sales/Lead summary/email: 140 sessions against a
              series median of 53 (2.6x). Notes say: 'traffic spike from demo
              account'.
  QUARANTINE volume collapse  [30 sessions]
              2026-08-07 Support/Reply draft/queue: 30 sessions against a
              series median of 64 (0.5x). Notes say: 'review policy changed
              mid-day'.
  WARN       incomplete day
              2026-08-05: 5/6 series usable. Missing Sales/Lead summary/email.
              Day-lev